# Synergy — first-time sign-in

Signs in to Synergy **once, in a window you can watch**, using the same browser profile the parser uses
(`parser/inputs/_browser_profile`). You type the emailed security code yourself, so nothing has to automate
two-factor.

After this succeeds, parser runs open that same profile and Synergy should recognise the device — no code,
no flood of emails.

**Before you run it:** stop any parser run that is going. Chromium locks a profile folder, so two at once
will fail (the parser falls back to a throwaway profile and you are back to being asked for a code).

Run cell 1, finish the sign-in in the window, then run cell 2 to prove it stuck.

In [3]:
# --- setup: paths, credentials, and a worker thread Playwright is happy on ---------------------------
import os, re, sys, time, asyncio, concurrent.futures

PARSER_DIR = os.getcwd()
if not os.path.isdir(os.path.join(PARSER_DIR, "parser_lib")):
    raise SystemExit(f"Run this notebook from the 'parser' folder (this is {PARSER_DIR}).")

INPUT_DIR   = os.path.join(PARSER_DIR, "inputs")
PROFILE_DIR = os.environ.get("FASTSCOUT_PROFILE_DIR") or os.path.join(INPUT_DIR, "_browser_profile")
START_URL    = "https://apps.synergysports.com/basketball"
WAIT_MINUTES = 10          # how long the window stays open waiting for you to finish

try:
    from dotenv import load_dotenv
    for _p in (os.path.join(PARSER_DIR, ".env"), os.path.join(os.path.dirname(PARSER_DIR), ".env")):
        if os.path.exists(_p):
            load_dotenv(_p, override=False)
except ModuleNotFoundError:
    pass

SYN_USER = os.environ.get("SYNERGY_USERNAME")
SYN_PASS = os.environ.get("SYNERGY_PASSWORD")
print("profile folder :", PROFILE_DIR, "(exists)" if os.path.isdir(PROFILE_DIR) else "(will be created)")
print("credentials    :", "found in .env" if (SYN_USER and SYN_PASS) else
      "NOT found -- you can still type them in the window yourself")


def run_on_worker(fn):
    """Playwright's sync API will not run on a thread that already has an asyncio loop, which is exactly
    what a Jupyter kernel's main thread has. The parser solves this the same way (run_in_fastscout_session):
    do the work on a plain worker thread, and swap in the Proactor loop policy Windows needs for
    Playwright's Node driver."""
    def _job():
        original = asyncio.get_event_loop_policy() if sys.platform == "win32" else None
        if sys.platform == "win32":
            asyncio.set_event_loop_policy(asyncio.WindowsProactorEventLoopPolicy())
        try:
            return fn()
        finally:
            if original is not None:
                asyncio.set_event_loop_policy(original)
    with concurrent.futures.ThreadPoolExecutor(max_workers=1) as ex:
        return ex.submit(_job).result()


def _left_identity_provider(url):
    return ("auth.synergysportstech.com" not in url) and ("/Account/" not in url)


profile folder : F:\uwwmensbball\parser\inputs\_browser_profile (exists)
credentials    : found in .env


In [ ]:
# --- 1. sign in, in a window you can see ------------------------------------------------------------
def _sign_in():
    from playwright.sync_api import sync_playwright
    pw = sync_playwright().start()
    ctx = None
    try:
        os.makedirs(PROFILE_DIR, exist_ok=True)
        ctx = pw.chromium.launch_persistent_context(PROFILE_DIR, headless=False,
                                                    args=["--start-maximized"], no_viewport=True)
        page = ctx.pages[0] if ctx.pages else ctx.new_page()
        page.goto(START_URL, wait_until="domcontentloaded", timeout=60000)
        page.wait_for_timeout(2500)

        if _left_identity_provider(page.url):
            print("Already signed in on this profile -- nothing to do.")
            return True

        # Fill what we can, so you only have to deal with the code itself.
        try:
            _u = page.locator('input[name="Username"], input[type="email"]').first
            if SYN_USER and SYN_PASS and _u.count():
                _u.fill(SYN_USER)
                page.locator('input[type="password"]').first.fill(SYN_PASS)
                _r = page.locator('input[type="checkbox"][name="RememberLogin"]').first
                if _r.count() and not _r.is_checked():
                    _r.check()                      # without this Synergy issues no device token at all
                page.get_by_role("button", name=re.compile(r"^log ?in$", re.I)).first.click()
                print("Submitted the saved credentials with 'Remember My Login' ticked.")
        except Exception as e:
            print(f"Could not fill the form ({type(e).__name__}: {e}) -- type it in the window yourself.")

        print(f"\nFinish in the browser window. Waiting up to {WAIT_MINUTES} min.")
        _deadline, _said = time.time() + WAIT_MINUTES * 60, False
        while time.time() < _deadline:
            _url = page.url
            if _left_identity_provider(_url):
                print("\nSigned in. This profile now holds Synergy's remembered-device token.")
                page.wait_for_timeout(3000)         # let the app set its cookies before we close
                return True
            if not _said and ("VerifyCode" in _url or page.locator("input#Code").count()):
                print("  -> Synergy emailed a code. Type it in the window and press Login. "
                      "Use the NEWEST email; earlier codes are dead.")
                _said = True
            time.sleep(2)
        print("\nTimed out still on the sign-in pages. Nothing was saved; run the cell again.")
        return False
    finally:
        for _obj, _closer in ((ctx, "close"), (pw, "stop")):
            try:
                getattr(_obj, _closer)()            # closing is what flushes the profile to disk
            except Exception:
                pass

ok = run_on_worker(_sign_in)
print("\nresult:", "signed in" if ok else "not signed in")


In [5]:
# --- 2. prove it stuck: open the SAME profile headless, the way the parser does ----------------------
def _verify():
    from playwright.sync_api import sync_playwright
    pw = sync_playwright().start()
    ctx = None
    try:
        ctx = pw.chromium.launch_persistent_context(PROFILE_DIR, headless=True)
        page = ctx.pages[0] if ctx.pages else ctx.new_page()
        page.goto(START_URL, wait_until="domcontentloaded", timeout=60000)
        for _ in range(15):
            if _left_identity_provider(page.url):
                return True, page.url
            page.wait_for_timeout(1000)
        return False, page.url
    finally:
        for _obj, _closer in ((ctx, "close"), (pw, "stop")):
            try:
                getattr(_obj, _closer)()
            except Exception:
                pass

good, where = run_on_worker(_verify)
print("headless, same profile ->", "still signed in: the parser will be too" if good else
      f"sent back to sign in ({where})")
if not good:
    print("If this says 'sent back to sign in', the token did not persist. Check that no other run holds\n"
          "the profile folder, then try cell 1 again.")


headless, same profile -> still signed in: the parser will be too
